## Computation of spherical harmonic coefficients


<a href="[text](https://creativecommons.org/licenses/by-nc/4.0/)">LaserIllumination</a> © 2025 by <a href="https://creativecommons.org">M. Cotelo, A. Lorca (Universidad Politécnica de Madrid)</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International</a><img src="https://mirrors.creativecommons.org/presskit/icons/cc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/by.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/nc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/sa.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;">


Authors:
- Manuel Cotelo Ferreiro (<manuel.cotelo@upm.es>) (Instituto de Fusión Nuclear Guillermo Velarde, Universidad Politécnica de Madrid)
- Alberto Lorca (<alberto.lorca@alumnos.upm.es>) (Universidad Politécnica de Madrid)

In [1]:
import numpy as np 
import pandas as pd 
from rwhist import compute_sph_coeffs, print_coeffs
from scipy.special import sph_harm

### Reading the histogram data

In [2]:
def load_histogram(filename, n_theta=50, n_phi=50, column="hs_density"):
    
    df = pd.read_hdf(filename)   # sin key="impactos"

    if column not in df.columns:
        raise KeyError(
            f"La columna '{column}' no existe. Columnas disponibles: {df.columns.tolist()}"
        )

    hist_2d = df[column].values.reshape((n_theta, n_phi))
    return hist_2d

hist = load_histogram("histograma_muestreo.h5", n_theta=50, n_phi=50, column="hs_density")


### Calculation of spherical harmonics

\begin{align}
a_{l,m} & = \int_{0}^{2 \pi} d \phi \int_{0}^{\pi} d \theta sin \theta f(\theta,\phi) Y_{l,m}^{*} (\theta,\phi) \\
& = \sum^{N} \int_{\phi_{j}}^{\phi_{j+1}} d \phi \int_{\theta_{i}}^{\theta_{i+1}} d \theta sin \theta f(\theta,\phi) Y_{l,m}^{*} (\theta,\phi) \\
\end{align}
 
Assume $f(\theta,\phi) = f_{i,j} = CTE$ for each sector $[\theta_{i},\theta_{i+1}] \times [\phi_{j},\phi_{j+1}]$
 
\begin{align}
a_{l,m} & = \sum^{N} f_{i,j} \int_{\phi_{j}}^{\phi_{j+1}} d \phi \int_{\theta_{i}}^{\theta_{i+1}} d \theta sin \theta Y_{l,m}^{*} (\theta,\phi) \\
& \simeq \sum^{N} f_{i,j} \cdot sin(\theta_{i + \frac{1}{2}}) \cdot Y_{l,m}^{*}(\theta_{i+\frac{1}{2}},\phi_{j+\frac{1}{2}}) \cdot S_{i,j}
\end{align}

### Physical approximation and calculation of spherical harmonics

In [3]:
def analyze_histogram(hist, theta_edges, phi_edges, Lmax=6):

    print("\n" + "="*80)
    print("=== ANALISIS DE ARMÓNICOS ESFÉRICOS DEL HISTOGRAMA REAL ===")
    print("="*80 + "\n")

    coeffs = compute_sph_coeffs(hist, theta_edges, phi_edges, Lmax)
    print_coeffs(coeffs)

    return coeffs


In [4]:

n_theta, n_phi = 50, 50
theta_edges = np.linspace(0, np.pi, n_theta + 1)
phi_edges = np.linspace(0, 2*np.pi, n_phi + 1)

# 4. Analizar armónicos
analyze_histogram(hist, theta_edges, phi_edges, Lmax=6)



=== ANALISIS DE ARMÓNICOS ESFÉRICOS DEL HISTOGRAMA REAL ===


================ COEFICIENTES NORMALIZADOS a_{lm} =================

  l    m        Re(a_lm)        Im(a_lm)          |a_lm|
------------------------------------------------------------
  0    0    2.820948e-01    0.000000e+00    2.820948e-01    1.000000e+00
  1   -1    4.591456e-03    4.068894e-03    6.134930e-03    2.174776e-02
  1    0    6.924274e-03    0.000000e+00    6.924274e-03    2.454591e-02
  1    1   -4.591456e-03    4.068894e-03    6.134930e-03    2.174776e-02
  2   -2   -1.356969e-02    2.516097e-03    1.380099e-02    4.892323e-02
  2   -1   -7.344219e-04   -5.958585e-03    6.003675e-03    2.128247e-02
  2    0    9.809501e-02    0.000000e+00    9.809501e-02    3.477378e-01
  2    1    7.344219e-04   -5.958585e-03    6.003675e-03    2.128247e-02
  2    2   -1.356969e-02   -2.516097e-03    1.380099e-02    4.892323e-02
  3   -3   -2.153543e-03    5.525985e-03    5.930789e-03    2.102410e-02
  3   -2   -7.174567

{(0, 0): np.complex128(0.2820947917738781+0j),
 (1, -1): np.complex128(0.004591456417572709+0.004068893755400405j),
 (1, 0): np.complex128(0.006924274397462111+0j),
 (1, 1): np.complex128(-0.004591456417572709+0.004068893755400405j),
 (2, -2): np.complex128(-0.013569692855079824+0.0025160967710245643j),
 (2, -1): np.complex128(-0.000734421898479215-0.0059585847400994265j),
 (2, 0): np.complex128(0.09809500866195675+0j),
 (2, 1): np.complex128(0.000734421898479215-0.0059585847400994265j),
 (2, 2): np.complex128(-0.013569692855079824-0.0025160967710245643j),
 (3, -3): np.complex128(-0.002153543028024538+0.005525984855068808j),
 (3, -2): np.complex128(-0.0007174567038645179-0.0036570868202956855j),
 (3, -1): np.complex128(0.007730192537675798+0.0039349836412508315j),
 (3, 0): np.complex128(0.0011792685215083307+0j),
 (3, 1): np.complex128(-0.007730192537675798+0.0039349836412508315j),
 (3, 2): np.complex128(-0.0007174567038645179+0.0036570868202956855j),
 (3, 3): np.complex128(0.002153543